# Prompt Tuning 交互教学

配套 lecture：[`../lectures/02-prompt-tuning.md`](../lectures/02-prompt-tuning.md)  
配套论文：[`../papers/02-prompt-tuning-2021.pdf`](../papers/02-prompt-tuning-2021.pdf)  
配套代码：[`../src/prompt_tuning_minimal.py`](../src/prompt_tuning_minimal.py) · [`../src/prompt_tuning_peft.py`](../src/prompt_tuning_peft.py)

本 notebook 逐步演示：
1. 环境检查
2. 手写版（minimal）构造与参数量
3. peft 调包版
4. 数值一致性验证
5. Toy 训练演示
6. 思考题

## 1. 环境检查

In [ ]:
import sys
import torch, transformers, peft
print(f'Python: {sys.version.split()[0]}')
print(f'torch:        {torch.__version__}, cuda={torch.cuda.is_available()}')
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')

## 2. 手写版（minimal）

公式 (1): $\widetilde{\mathbf{E}} = [\mathbf{P}; \mathbf{E}]$

其中 $\mathbf{P} \in \mathbb{R}^{p \times d}$ 是唯一可训练参数。

In [ ]:
import sys
from pathlib import Path
# 把 src/ 加入 sys.path，方便 import minimal/peft
src_dir = (Path.cwd().parent / 'src').resolve()
sys.path.insert(0, str(src_dir))
print(f'src dir: {src_dir}')

from prompt_tuning_minimal import PromptTuningGPT2
from common import print_param_summary

torch.manual_seed(42)
model = PromptTuningGPT2(prompt_length=10)
print_param_summary(model, 'minimal')
# 预期：可训练 = 10 * 768 = 7,680

In [ ]:
# 查看 P 的具体形状
print(f'P shape: {tuple(model.prompt_embeddings.shape)}')
print(f'P 第 1 行前 10 个值: {model.prompt_embeddings[0, :10].tolist()}')

## 3. peft 调包版

用 `peft.PromptTuningConfig` 实现同一算法。

In [ ]:
from prompt_tuning_peft import build_peft_model

torch.manual_seed(42)
peft_model = build_peft_model(prompt_length=10)
print_param_summary(peft_model, 'peft')

# 打印 peft 内部存储
print('\npeft 可训练参数清单：')
for name, p in peft_model.named_parameters():
    if p.requires_grad:
        print(f'  {name}: shape={tuple(p.shape)}')

## 4. 数值一致性验证

把 minimal 的 $\mathbf{P}$ 复制到 peft，验证 logits 数值一致。

In [ ]:
sys.path.insert(0, str((Path.cwd().parent / 'src' / 'tests').resolve()))
from test_prompt_consistency import test_logits_match
test_logits_match()

## 5. Toy 训练演示

用玩具情感数据集跑 5 步训练，观察 loss 下降。

注意：这只是流程演示，不追求 loss 收敛。

In [ ]:
from prompt_tuning_minimal import toy_train
torch.manual_seed(42)
model_for_train = PromptTuningGPT2(prompt_length=10)
toy_train(model_for_train, num_steps=5)

## 6. 思考题

1. **参数量预测**：若把 `prompt_length` 从 10 改成 100，新增可训练参数是多少？请预测后验证。
2. **初始化对比**：用 `init_text='positive negative'` 初始化 $\mathbf{P}$，与随机初始化在 toy 数据集上跑 50 步，loss 曲线会如何不同？
3. **peft 内部**：打印 `peft_model.prompt_encoder.default`，它是什么类？
4. **小模型实验**：本 notebook 用的是 `gpt2`（117M），论文 Figure 1 提示：在这个规模上 Prompt Tuning 应该比全参微调差很多。这与论文中 T5-small（60M）的结论一致吗？
5. **多任务存储**：若有 10 个任务，分别用 Prompt Tuning，总额外存储多少 MB？与全参微调（每任务 117M × 4 bytes ≈ 470 MB）对比。

In [ ]:
# 提示：思考题 1 的验证代码
torch.manual_seed(0)
m_big = PromptTuningGPT2(prompt_length=100)
print_param_summary(m_big, 'p=100')
# 期望 trainable = 100 * 768 = 76,800

In [ ]:
# 提示：思考题 3 的验证代码
print(type(peft_model.prompt_encoder.default).__name__)
print(peft_model.prompt_encoder.default)